# 📈 Econometric Diagnostics & Remedies: Non-constant Variance and Serial Correlation
### *A Comprehensive Rigorous Computational Guide to Gauss-Markov Assumption Violations*
**Author:** Vaibhav ([@Vaibhav007-star](https://github.com/Vaibhav007-star))

---
## 📌 Theoretical Background
Under the classical Gauss-Markov assumptions, Ordinary Least Squares (OLS) produces **Best Linear Unbiased Estimators (BLUE)**.
However, two critical assumptions are frequently violated in real-world cross-sectional and time-series data:

1. **Homoscedasticity:** $\text{Var}(\epsilon_i | X) = \sigma^2$
   - **Violation:** Heteroscedasticity ($\text{Var}(\epsilon_i | X) = \sigma_i^2$).
   - **Consequence:** OLS point estimates remain unbiased and consistent, but standard errors are biased, rendering hypothesis tests ($t$-tests, $F$-tests) invalid.

2. **No Autocorrelation:** $\text{Cov}(\epsilon_t, \epsilon_s | X) = 0 \quad \forall t \neq s$
   - **Violation:** Serial Correlation (e.g. $AR(1): \epsilon_t = \rho \epsilon_{t-1} + u_t$).
   - **Consequence:** OLS standard errors are severely underestimated, producing artificially inflated $t$-statistics and false discovery of statistical significance.

In [ ]:
import sys
import os
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm

from src.pipeline.econometric_engine import EconometricAuditEngine
from src.visualizations.diagnostic_plots import (
    plot_residuals_vs_fitted,
    plot_scale_location,
    plot_autocorrelation,
    plot_lag_scatter
)

print("All modules loaded successfully!")

## 1. Case Study: Cross-Sectional Housing Data (Heteroscedasticity)
Let's load `housing_prices.csv` where error variance expands as house size increases.

In [ ]:
df_housing = pd.read_csv('../data/housing_prices.csv')
engine_housing = EconometricAuditEngine(
    data=df_housing,
    target_col='Price',
    feature_cols=['Square_Feet', 'Lot_Size', 'Bedrooms', 'Age_Years', 'School_Rating']
)
model_h = engine_housing.fit_baseline()
print(model_h.summary())

In [ ]:
# Diagnostic Plots
fig = plot_residuals_vs_fitted(model_h)
plt.show()

In [ ]:
# Formal Heteroscedasticity Tests
diag_h = engine_housing.run_diagnostics()
import pprint
pprint.pprint(diag_h['heteroscedasticity'])

## 2. Case Study: Macroeconomic Time-Series (Serial Correlation & ARCH)
Now let's examine `macroeconomic_ts.csv` where quarterly GDP growth exhibits high persistence.

In [ ]:
df_macro = pd.read_csv('../data/macroeconomic_ts.csv')
engine_macro = EconometricAuditEngine(
    data=df_macro,
    target_col='GDP_Growth',
    feature_cols=['Interest_Rate', 'Money_Growth', 'Unemployment_Rate']
)
model_m = engine_macro.fit_baseline()
fig_acf = plot_autocorrelation(model_m)
plt.show()
fig_lag = plot_lag_scatter(model_m)
plt.show()

## 3. Comparative Remedies Scorecard
Comparing baseline OLS vs. White HC3 vs. Newey-West HAC vs. Feasible GLS.

In [ ]:
scorecard = engine_macro.generate_scorecard()
scorecard